# Step 12 – GraphSAGE: Predicting Adoption Intensity (Revised)

**RQ3**: *Can the city collaboration network structure, combined with city attributes,
predict which cities will be high-intensity adopters of new open-AI projects?*

### Approach

| Model | Target | Method |
|---|---|---|
| G1 | `log(test_adoption_count)` | GraphSAGE regression |
| G2 | `is_high_adopter` (above median) | GraphSAGE classification |
| Ablation | same targets | MLP (no graph structure) |

### Key improvements over initial version

1. **Feature leakage fix** — removed full-period `adoption_count`, `origination_count`,
   `weighted_degree` etc. from features; activity features are now computed only from
   the training period (≤ 202406).
2. **Train / Val / Test split** — stratified 60 / 20 / 20 (instead of 70 / 30 without
   validation set).
3. **Regularisation** — dropout raised to 0.5, max epochs reduced to 150 with
   early-stopping (patience = 20) on validation loss/AUC.
4. **MLP ablation** — an identically-sized MLP baseline quantifies how much value the
   graph structure actually adds.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from sklearn.metrics import (roc_auc_score, f1_score, classification_report,
                             mean_squared_error, r2_score, accuracy_score)
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedShuffleSplit
from scipy.stats import spearmanr

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({'figure.dpi': 120})
torch.manual_seed(42)
np.random.seed(42)

DATA = Path('..') / 'data' / 'output'
city_attr   = pd.read_csv(DATA / 'city_attributes.csv')
edges_month = pd.read_csv(DATA / 'city_collaboration_edges_monthly.csv')
adoption    = pd.read_csv(DATA / 'city_project_adoption_events.csv')

city_attr_dedup = city_attr.drop_duplicates(subset='city', keep='first')
print(f'Cities: {len(city_attr_dedup)}, Monthly edges: {len(edges_month)}, '
      f'Adoptions: {len(adoption)}')

## 12.1 – Temporal Split & Label Construction

In [ ]:
CUTOFF = 202406

edge_cities  = set(edges_month['source_city']) | set(edges_month['target_city'])
adopt_cities = set(adoption['city'])
all_cities   = sorted(edge_cities | adopt_cities)
city2idx     = {c: i for i, c in enumerate(all_cities)}
n_cities     = len(all_cities)

train_adoptions = adoption[adoption['city_first_adoption_month'] <= CUTOFF]
test_adoptions  = adoption[adoption['city_first_adoption_month'] >  CUTOFF]

# Regression target: log(test adoption count + 1)
test_counts = test_adoptions.groupby('city').size().to_dict()
y_reg = torch.zeros(n_cities, dtype=torch.float)
for city, idx in city2idx.items():
    y_reg[idx] = np.log1p(test_counts.get(city, 0))

# Classification target: above-median adopter
test_count_vals = [test_counts.get(c, 0) for c in all_cities]
median_adopt = np.median([v for v in test_count_vals if v > 0])
y_cls = torch.tensor(
    [1 if v >= median_adopt else 0 for v in test_count_vals], dtype=torch.long)

print(f'Active cities: {n_cities}')
print(f'Train adoptions: {len(train_adoptions)}, Test adoptions: {len(test_adoptions)}')
print(f'Test adoption median (among active): {median_adopt:.0f}')
print(f'High adopters (>=median): {y_cls.sum().item()}, '
      f'Low adopters: {(y_cls==0).sum().item()}')

## 12.2 – Build Graph (Leakage-Aware Features)

Feature engineering is split into two groups:

| Group | Features | Leakage risk |
|---|---|---|
| **A – Exogenous** | population, GDP, education, internet, R&D, research capacity | None (time-invariant) |
| **B – Activity** | train-period adoption / origination counts, train-period weighted degree, train-period entity count | None (computed only from data ≤ cutoff) |

The original notebook used **full-period** `adoption_count`, `origination_count`,
`weighted_degree` etc. from `city_attributes.csv` — these leak test-period information
into features and are now replaced with train-period-only versions.

In [ ]:
# ── Edges: train period only ─────────────────────────────────
train_edges = edges_month[edges_month['month'] <= CUTOFF]
train_agg = (train_edges.groupby(['source_city', 'target_city'])
             .agg(weight=('edge_weight', 'sum')).reset_index())

src_idx = [city2idx[c] for c in train_agg['source_city'] if c in city2idx]
tgt_idx = [city2idx[c] for c in train_agg['target_city'] if c in city2idx]
edge_index = torch.tensor([src_idx + tgt_idx, tgt_idx + src_idx], dtype=torch.long)

# ── Group A: Exogenous / time-invariant features ─────────────
exo_cols = ['population_million', 'gdp_per_capita',
            'education_tertiary_pct', 'internet_users_pct',
            'rd_expenditure_pct', 'research_capacity']

city_feat_map = city_attr_dedup.set_index('city')[exo_cols].to_dict('index')
feat_exo = np.zeros((n_cities, len(exo_cols)))
for city, idx in city2idx.items():
    if city in city_feat_map:
        for j, col in enumerate(exo_cols):
            val = city_feat_map[city].get(col, 0)
            feat_exo[idx, j] = val if not np.isnan(val) else 0

# ── Group B: Activity features (train period only) ───────────
train_adopt_ct  = train_adoptions.groupby('city').size().to_dict()
train_orig_ct   = (train_adoptions[train_adoptions['is_originator'] == 1]
                   .groupby('city').size().to_dict())
train_entity_ct = train_adoptions.groupby('city')['project_id'].nunique().to_dict()

train_degree = {}
for _, row in train_agg.iterrows():
    s, t, w = row['source_city'], row['target_city'], row['weight']
    train_degree[s] = train_degree.get(s, 0) + w
    train_degree[t] = train_degree.get(t, 0) + w

activity_names = ['train_adopt_count', 'train_orig_count',
                  'train_weighted_degree', 'train_entity_count']
feat_activity = np.zeros((n_cities, len(activity_names)))
for city, idx in city2idx.items():
    feat_activity[idx, 0] = train_adopt_ct.get(city, 0)
    feat_activity[idx, 1] = train_orig_ct.get(city, 0)
    feat_activity[idx, 2] = train_degree.get(city, 0)
    feat_activity[idx, 3] = train_entity_ct.get(city, 0)

# ── Combine & scale ──────────────────────────────────────────
all_feats  = np.hstack([feat_exo, feat_activity])
feat_names = exo_cols + activity_names

scaler = StandardScaler()
x = torch.tensor(scaler.fit_transform(all_feats), dtype=torch.float)

print(f'Edges: {edge_index.shape[1]//2} undirected, Nodes: {n_cities}')
print(f'Features: {x.shape[1]} dims  '
      f'({len(exo_cols)} exogenous + {len(activity_names)} train-period activity)')
print(f'Feature columns: {feat_names}')

## 12.3 – Feature Leakage Diagnostic

In [ ]:
feat_df = pd.DataFrame(all_feats, columns=feat_names)
feat_df['y_log_adopt'] = y_reg.numpy()

corr_new = (feat_df.corr()['y_log_adopt']
            .drop('y_log_adopt').sort_values(ascending=False))

print('Current feature correlations with test-period log(adoption):')
for name, val in corr_new.items():
    flag = '  ⚠ HIGH' if abs(val) > 0.8 else ''
    print(f'  {name:30s}: r = {val:+.3f}{flag}')

# Compare with the OLD full-period features that had potential leakage
old_leaky_cols = ['adoption_count', 'origination_count',
                  'entity_count', 'weighted_degree']
old_corr = {}
for col in old_leaky_cols:
    if col in city_attr_dedup.columns:
        m = city_attr_dedup.set_index('city')[col].to_dict()
        vals = np.array([m.get(c, 0) for c in all_cities])
        r = np.corrcoef(vals, y_reg.numpy())[0, 1]
        old_corr[col] = r

if old_corr:
    print('\n⚠ OLD full-period features (removed — potential leakage):')
    for col, r in old_corr.items():
        print(f'  {col:30s}: r = {r:+.3f}')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

colors = ['#2166ac' if abs(v) < 0.7 else '#d73027' for v in corr_new.values]
corr_new.plot.barh(ax=axes[0], color=colors)
axes[0].axvline( 0.7, color='red', ls='--', alpha=.4, label='|r|=0.7')
axes[0].axvline(-0.7, color='red', ls='--', alpha=.4)
axes[0].set_xlabel('Pearson r with log(test adoption)')
axes[0].set_title('Current Features (leakage-aware)')
axes[0].legend()

if old_corr:
    all_labels = list(corr_new.index) + list(old_corr.keys())
    all_vals   = list(corr_new.values) + list(old_corr.values())
    src_label  = (['Current'] * len(corr_new) +
                  ['OLD (removed)'] * len(old_corr))
    cmp = pd.DataFrame({'feature': all_labels, 'r': all_vals, 'source': src_label})
    cmp = cmp.sort_values('r', ascending=True)
    palette = {'Current': '#2166ac', 'OLD (removed)': '#d73027'}
    sns.barplot(data=cmp, y='feature', x='r', hue='source',
                palette=palette, ax=axes[1], dodge=False)
    axes[1].axvline(0.7, color='red', ls='--', alpha=.4)
    axes[1].set_xlabel('Pearson r with log(test adoption)')
    axes[1].set_title('Current vs OLD Features')

plt.tight_layout()
plt.show()

## 12.4 – Train / Val / Test Split

Stratified 60 / 20 / 20 split based on classification label (`y_cls`) to ensure
balanced high-/low-adopter representation in every subset.  A dedicated **validation
set** enables early stopping and honest hyper-parameter selection.

In [ ]:
idx_arr = np.arange(n_cities)
y_strat = y_cls.numpy()

sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.4, random_state=42)
train_idx, temp_idx = next(sss1.split(idx_arr, y_strat))

sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
val_rel, test_rel = next(sss2.split(temp_idx, y_strat[temp_idx]))
val_idx  = temp_idx[val_rel]
test_idx = temp_idx[test_rel]

train_mask = torch.zeros(n_cities, dtype=torch.bool)
val_mask   = torch.zeros(n_cities, dtype=torch.bool)
test_mask  = torch.zeros(n_cities, dtype=torch.bool)
train_mask[train_idx] = True
val_mask[val_idx]     = True
test_mask[test_idx]   = True

for name, mask in [('Train', train_mask), ('Val', val_mask), ('Test', test_mask)]:
    n = mask.sum().item()
    high = y_cls[mask].sum().item()
    print(f'{name:5s}: {n:3d} nodes  '
          f'(high adopter: {high}/{n} = {high/n:.1%})')

## 12.5 – Model G1: GraphSAGE Regression

Changes from initial version:
- **Dropout 0.3 → 0.5**, **weight decay 5e-4 → 1e-3**
- **Max epochs 300 → 150** with **early stopping** (patience = 20 on val MSE)
- Validation-set monitoring prevents overfitting the small graph (148 nodes)

In [ ]:
class GraphSAGE_Reg(nn.Module):
    def __init__(self, in_ch, hidden, dropout=0.5):
        super().__init__()
        self.conv1 = SAGEConv(in_ch, hidden)
        self.conv2 = SAGEConv(hidden, hidden)
        self.head  = nn.Linear(hidden, 1)
        self.drop  = dropout

    def forward(self, x, ei):
        h = F.relu(self.conv1(x, ei))
        h = F.dropout(h, p=self.drop, training=self.training)
        h = F.relu(self.conv2(h, ei))
        h = F.dropout(h, p=self.drop, training=self.training)
        return self.head(h).squeeze(-1)

    def get_embeddings(self, x, ei):
        h = F.relu(self.conv1(x, ei))
        h = F.relu(self.conv2(h, ei))
        return h

MAX_EPOCHS = 150
PATIENCE   = 20
HIDDEN     = 64
DROPOUT    = 0.5
LR         = 0.01
WD         = 1e-3

model_reg = GraphSAGE_Reg(x.shape[1], HIDDEN, dropout=DROPOUT)
optimizer = torch.optim.Adam(model_reg.parameters(), lr=LR, weight_decay=WD)

best_val_loss = float('inf')
best_state    = None
patience_ctr  = 0
train_losses, val_losses = [], []

for epoch in range(1, MAX_EPOCHS + 1):
    model_reg.train()
    optimizer.zero_grad()
    out  = model_reg(x, edge_index)
    loss = F.mse_loss(out[train_mask], y_reg[train_mask])
    loss.backward()
    optimizer.step()

    model_reg.eval()
    with torch.no_grad():
        pred     = model_reg(x, edge_index)
        val_loss = F.mse_loss(pred[val_mask], y_reg[val_mask]).item()

    train_losses.append(loss.item())
    val_losses.append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.clone() for k, v in model_reg.state_dict().items()}
        patience_ctr = 0
    else:
        patience_ctr += 1

    if epoch % 25 == 0:
        tr_r2 = r2_score(y_reg[train_mask].numpy(), pred[train_mask].numpy())
        vl_r2 = r2_score(y_reg[val_mask].numpy(),   pred[val_mask].numpy())
        te_r2 = r2_score(y_reg[test_mask].numpy(),  pred[test_mask].numpy())
        print(f'Epoch {epoch:3d}  loss={loss.item():.4f}  val_loss={val_loss:.4f}  '
              f'train_R²={tr_r2:.3f}  val_R²={vl_r2:.3f}  test_R²={te_r2:.3f}')

    if patience_ctr >= PATIENCE:
        print(f'\n✓ Early stopping at epoch {epoch} (patience={PATIENCE})')
        break

model_reg.load_state_dict(best_state)
model_reg.eval()
with torch.no_grad():
    pred_reg = model_reg(x, edge_index)

gs_metrics_reg = {}
print(f'\nModel G1 – GraphSAGE Regression (best val epoch):')
for name, mask in [('Train', train_mask), ('Val', val_mask), ('Test', test_mask)]:
    r2   = r2_score(y_reg[mask].numpy(), pred_reg[mask].numpy())
    rmse = np.sqrt(mean_squared_error(y_reg[mask].numpy(), pred_reg[mask].numpy()))
    gs_metrics_reg[name] = {'R2': r2, 'RMSE': rmse}
    print(f'  {name:5s}  R²={r2:.4f}  RMSE={rmse:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss curves
axes[0].plot(train_losses, color='steelblue', lw=.8, label='Train')
axes[0].plot(val_losses,   color='darkorange', lw=.8, label='Val')
stop_ep = len(train_losses) - patience_ctr if patience_ctr >= PATIENCE else len(train_losses)
axes[0].axvline(stop_ep, color='red', ls='--', alpha=.5,
                label=f'Best epoch ({stop_ep})')
axes[0].set(xlabel='Epoch', ylabel='MSE Loss', title='G1: Train & Val Loss')
axes[0].legend()

# Actual vs Predicted
ya = y_reg.numpy()
yp = pred_reg.detach().numpy()
axes[1].scatter(ya[train_mask], yp[train_mask], alpha=.5, c='steelblue',
                s=30, label=f'Train (R²={gs_metrics_reg["Train"]["R2"]:.3f})')
axes[1].scatter(ya[val_mask],   yp[val_mask],   alpha=.7, c='green',
                s=50, marker='s', label=f'Val (R²={gs_metrics_reg["Val"]["R2"]:.3f})')
axes[1].scatter(ya[test_mask],  yp[test_mask],  alpha=.8, c='darkorange',
                s=60, edgecolor='k', label=f'Test (R²={gs_metrics_reg["Test"]["R2"]:.3f})')
lims = [min(ya.min(), yp.min()) - .5, max(ya.max(), yp.max()) + .5]
axes[1].plot(lims, lims, 'k--', alpha=.5, lw=1)
axes[1].set(xlabel='Actual log(adoption)', ylabel='Predicted', title='G1: Actual vs Predicted')
axes[1].legend()

# Residual
res = yp - ya
axes[2].scatter(yp[train_mask], res[train_mask], alpha=.5, c='steelblue', s=30, label='Train')
axes[2].scatter(yp[val_mask],   res[val_mask],   alpha=.7, c='green', s=50, marker='s', label='Val')
axes[2].scatter(yp[test_mask],  res[test_mask],  alpha=.8, c='darkorange', s=60, edgecolor='k', label='Test')
axes[2].axhline(0, color='k', ls='--', alpha=.5)
axes[2].set(xlabel='Predicted', ylabel='Residual', title='G1: Residual Plot')
axes[2].legend()

plt.tight_layout()
plt.show()

## 12.6 – Model G2: GraphSAGE Classification (High vs Low Adopter)

Same architectural & regularisation changes as G1.  Early stopping is based on
**validation AUC** (higher is better).

In [ ]:
class GraphSAGE_Cls(nn.Module):
    def __init__(self, in_ch, hidden, dropout=0.5):
        super().__init__()
        self.conv1 = SAGEConv(in_ch, hidden)
        self.conv2 = SAGEConv(hidden, hidden)
        self.head  = nn.Linear(hidden, 2)
        self.drop  = dropout

    def forward(self, x, ei):
        h = F.relu(self.conv1(x, ei))
        h = F.dropout(h, p=self.drop, training=self.training)
        h = F.relu(self.conv2(h, ei))
        h = F.dropout(h, p=self.drop, training=self.training)
        return self.head(h)

    def get_embeddings(self, x, ei):
        h = F.relu(self.conv1(x, ei))
        h = F.relu(self.conv2(h, ei))
        return h

n_pos = y_cls[train_mask].sum().item()
n_neg = train_mask.sum().item() - n_pos
cls_weights = torch.tensor([1.0, n_neg / max(n_pos, 1)], dtype=torch.float)

model_cls  = GraphSAGE_Cls(x.shape[1], HIDDEN, dropout=DROPOUT)
optimizer2 = torch.optim.Adam(model_cls.parameters(), lr=LR, weight_decay=WD)
criterion  = nn.CrossEntropyLoss(weight=cls_weights)

best_val_auc   = 0.0
best_state_cls = None
patience_cls   = 0

for epoch in range(1, MAX_EPOCHS + 1):
    model_cls.train()
    optimizer2.zero_grad()
    out  = model_cls(x, edge_index)
    loss = criterion(out[train_mask], y_cls[train_mask])
    loss.backward()
    optimizer2.step()

    model_cls.eval()
    with torch.no_grad():
        logits = model_cls(x, edge_index)
        probs  = F.softmax(logits, dim=1)[:, 1]
        v_y = y_cls[val_mask].numpy()
        v_p = probs[val_mask].numpy()
        val_auc = (roc_auc_score(v_y, v_p)
                   if len(np.unique(v_y)) > 1 else 0.5)

    if val_auc > best_val_auc:
        best_val_auc   = val_auc
        best_state_cls = {k: v.clone() for k, v in model_cls.state_dict().items()}
        patience_cls   = 0
    else:
        patience_cls += 1

    if epoch % 25 == 0:
        te_y = y_cls[test_mask].numpy()
        te_p = probs[test_mask].numpy()
        te_auc = roc_auc_score(te_y, te_p) if len(np.unique(te_y)) > 1 else 0
        f1_val = f1_score(te_y, logits.argmax(1)[test_mask].numpy(), zero_division=0)
        print(f'Epoch {epoch:3d}  loss={loss.item():.4f}  '
              f'val_AUC={val_auc:.4f}  test_AUC={te_auc:.4f}  test_F1={f1_val:.4f}')

    if patience_cls >= PATIENCE:
        print(f'\n✓ Early stopping at epoch {epoch} (patience={PATIENCE})')
        break

model_cls.load_state_dict(best_state_cls)
model_cls.eval()
with torch.no_grad():
    logits_gs = model_cls(x, edge_index)
    probs_gs  = F.softmax(logits_gs, dim=1)[:, 1]
    preds_gs  = logits_gs.argmax(1)

gs_metrics_cls = {}
for name, mask in [('Train', train_mask), ('Val', val_mask), ('Test', test_mask)]:
    y_t    = y_cls[mask].numpy()
    y_p    = preds_gs[mask].numpy()
    y_prob = probs_gs[mask].numpy()
    auc_v  = roc_auc_score(y_t, y_prob) if len(np.unique(y_t)) > 1 else float('nan')
    f1_v   = f1_score(y_t, y_p, zero_division=0)
    acc_v  = accuracy_score(y_t, y_p)
    gs_metrics_cls[name] = {'AUC': auc_v, 'F1': f1_v, 'Acc': acc_v}
    print(f'\n{name}: AUC={auc_v:.4f}  Acc={acc_v:.4f}  F1={f1_v:.4f}')
    print(classification_report(
        y_t, y_p, target_names=['Low adopter', 'High adopter'], zero_division=0))

## 12.7 – Ablation: MLP Baseline (No Graph Structure)

To quantify the **marginal value of graph propagation**, we train an identically-sized
2-layer MLP that receives the same node features but does **not** use the edge
structure.  If the MLP matches GraphSAGE's performance, the collaboration graph
contributes little additional predictive power beyond the node features alone.

In [ ]:
class MLP_Reg(nn.Module):
    def __init__(self, in_ch, hidden, dropout=0.5):
        super().__init__()
        self.fc1  = nn.Linear(in_ch, hidden)
        self.fc2  = nn.Linear(hidden, hidden)
        self.head = nn.Linear(hidden, 1)
        self.drop = dropout

    def forward(self, x):
        h = F.relu(self.fc1(x))
        h = F.dropout(h, p=self.drop, training=self.training)
        h = F.relu(self.fc2(h))
        h = F.dropout(h, p=self.drop, training=self.training)
        return self.head(h).squeeze(-1)

mlp_reg  = MLP_Reg(x.shape[1], HIDDEN, dropout=DROPOUT)
opt_mlp  = torch.optim.Adam(mlp_reg.parameters(), lr=LR, weight_decay=WD)

best_vl_mlp  = float('inf')
best_st_mlp  = None
pat_mlp      = 0
mlp_tr_loss, mlp_vl_loss = [], []

for epoch in range(1, MAX_EPOCHS + 1):
    mlp_reg.train()
    opt_mlp.zero_grad()
    out  = mlp_reg(x)
    loss = F.mse_loss(out[train_mask], y_reg[train_mask])
    loss.backward()
    opt_mlp.step()

    mlp_reg.eval()
    with torch.no_grad():
        pred    = mlp_reg(x)
        vl_loss = F.mse_loss(pred[val_mask], y_reg[val_mask]).item()

    mlp_tr_loss.append(loss.item())
    mlp_vl_loss.append(vl_loss)

    if vl_loss < best_vl_mlp:
        best_vl_mlp = vl_loss
        best_st_mlp = {k: v.clone() for k, v in mlp_reg.state_dict().items()}
        pat_mlp = 0
    else:
        pat_mlp += 1

    if epoch % 25 == 0:
        tr2 = r2_score(y_reg[train_mask].numpy(), pred[train_mask].numpy())
        vr2 = r2_score(y_reg[val_mask].numpy(),   pred[val_mask].numpy())
        tr2t = r2_score(y_reg[test_mask].numpy(), pred[test_mask].numpy())
        print(f'MLP Epoch {epoch:3d}  loss={loss.item():.4f}  '
              f'val_loss={vl_loss:.4f}  train_R²={tr2:.3f}  '
              f'val_R²={vr2:.3f}  test_R²={tr2t:.3f}')

    if pat_mlp >= PATIENCE:
        print(f'MLP early stopping at epoch {epoch}')
        break

mlp_reg.load_state_dict(best_st_mlp)
mlp_reg.eval()
with torch.no_grad():
    pred_mlp_reg = mlp_reg(x)

mlp_metrics_reg = {}
print('\nMLP Regression Results:')
for name, mask in [('Train', train_mask), ('Val', val_mask), ('Test', test_mask)]:
    r2   = r2_score(y_reg[mask].numpy(), pred_mlp_reg[mask].numpy())
    rmse = np.sqrt(mean_squared_error(y_reg[mask].numpy(), pred_mlp_reg[mask].numpy()))
    mlp_metrics_reg[name] = {'R2': r2, 'RMSE': rmse}
    print(f'  {name:5s}  R²={r2:.4f}  RMSE={rmse:.4f}')

In [ ]:
class MLP_Cls(nn.Module):
    def __init__(self, in_ch, hidden, dropout=0.5):
        super().__init__()
        self.fc1  = nn.Linear(in_ch, hidden)
        self.fc2  = nn.Linear(hidden, hidden)
        self.head = nn.Linear(hidden, 2)
        self.drop = dropout

    def forward(self, x):
        h = F.relu(self.fc1(x))
        h = F.dropout(h, p=self.drop, training=self.training)
        h = F.relu(self.fc2(h))
        h = F.dropout(h, p=self.drop, training=self.training)
        return self.head(h)

mlp_cls     = MLP_Cls(x.shape[1], HIDDEN, dropout=DROPOUT)
opt_mlp_cls = torch.optim.Adam(mlp_cls.parameters(), lr=LR, weight_decay=WD)
crit_mlp    = nn.CrossEntropyLoss(weight=cls_weights)

best_va_mlpc = 0.0
best_st_mlpc = None
pat_mlpc     = 0

for epoch in range(1, MAX_EPOCHS + 1):
    mlp_cls.train()
    opt_mlp_cls.zero_grad()
    out  = mlp_cls(x)
    loss = crit_mlp(out[train_mask], y_cls[train_mask])
    loss.backward()
    opt_mlp_cls.step()

    mlp_cls.eval()
    with torch.no_grad():
        logits_m = mlp_cls(x)
        probs_m  = F.softmax(logits_m, dim=1)[:, 1]
        v_y = y_cls[val_mask].numpy()
        v_p = probs_m[val_mask].numpy()
        va_auc = roc_auc_score(v_y, v_p) if len(np.unique(v_y)) > 1 else 0.5

    if va_auc > best_va_mlpc:
        best_va_mlpc = va_auc
        best_st_mlpc = {k: v.clone() for k, v in mlp_cls.state_dict().items()}
        pat_mlpc = 0
    else:
        pat_mlpc += 1

    if epoch % 25 == 0:
        te_y = y_cls[test_mask].numpy()
        te_p = probs_m[test_mask].numpy()
        te_auc = roc_auc_score(te_y, te_p) if len(np.unique(te_y)) > 1 else 0
        f1_v = f1_score(te_y, logits_m.argmax(1)[test_mask].numpy(), zero_division=0)
        print(f'MLP Epoch {epoch:3d}  loss={loss.item():.4f}  '
              f'val_AUC={va_auc:.4f}  test_AUC={te_auc:.4f}  test_F1={f1_v:.4f}')

    if pat_mlpc >= PATIENCE:
        print(f'MLP cls early stopping at epoch {epoch}')
        break

mlp_cls.load_state_dict(best_st_mlpc)
mlp_cls.eval()
with torch.no_grad():
    logits_m = mlp_cls(x)
    probs_m  = F.softmax(logits_m, dim=1)[:, 1]
    preds_m  = logits_m.argmax(1)

mlp_metrics_cls = {}
for name, mask in [('Train', train_mask), ('Val', val_mask), ('Test', test_mask)]:
    y_t    = y_cls[mask].numpy()
    y_p    = preds_m[mask].numpy()
    y_prob = probs_m[mask].numpy()
    auc_v  = roc_auc_score(y_t, y_prob) if len(np.unique(y_t)) > 1 else float('nan')
    f1_v   = f1_score(y_t, y_p, zero_division=0)
    acc_v  = accuracy_score(y_t, y_p)
    mlp_metrics_cls[name] = {'AUC': auc_v, 'F1': f1_v, 'Acc': acc_v}
    print(f'\nMLP {name}: AUC={auc_v:.4f}  Acc={acc_v:.4f}  F1={f1_v:.4f}')
    print(classification_report(
        y_t, y_p, target_names=['Low adopter', 'High adopter'], zero_division=0))

## 12.8 – GraphSAGE vs MLP Comparison

This section directly compares the two models on **test-set** metrics.  A meaningful
improvement by GraphSAGE over MLP would justify the added complexity of graph-based
learning; otherwise, the graph propagation provides little benefit for this small
network (148 nodes).

In [ ]:
comparison = pd.DataFrame({
    'Model':         ['GraphSAGE', 'MLP (no graph)'],
    'Reg Test R²':   [gs_metrics_reg['Test']['R2'],  mlp_metrics_reg['Test']['R2']],
    'Reg Test RMSE': [gs_metrics_reg['Test']['RMSE'], mlp_metrics_reg['Test']['RMSE']],
    'Cls Test AUC':  [gs_metrics_cls['Test']['AUC'],  mlp_metrics_cls['Test']['AUC']],
    'Cls Test F1':   [gs_metrics_cls['Test']['F1'],   mlp_metrics_cls['Test']['F1']],
}).round(4)

print('=' * 70)
print('ABLATION: GraphSAGE vs MLP (same features, same split)')
print('=' * 70)
display(comparison)

r2_diff  = gs_metrics_reg['Test']['R2']  - mlp_metrics_reg['Test']['R2']
auc_diff = gs_metrics_cls['Test']['AUC'] - mlp_metrics_cls['Test']['AUC']

if r2_diff > 0.05:
    verdict = (f'GraphSAGE outperforms MLP by ΔR²={r2_diff:+.4f}  — '
               f'graph structure adds meaningful value.')
elif r2_diff < -0.05:
    verdict = (f'MLP outperforms GraphSAGE by ΔR²={-r2_diff:+.4f}  — '
               f'graph propagation may hurt with so few nodes.')
else:
    verdict = (f'GraphSAGE ≈ MLP (ΔR²={r2_diff:+.4f})  — '
               f'graph structure provides limited additional value.')

print(f'\nRegression verdict: {verdict}')
print(f'Classification: ΔAUC = {auc_diff:+.4f},  ΔF1 = '
      f'{gs_metrics_cls["Test"]["F1"] - mlp_metrics_cls["Test"]["F1"]:+.4f}')

# ── Visual comparison ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

y_act = y_reg[test_mask].numpy()
axes[0].scatter(y_act, pred_reg[test_mask].detach().numpy(), alpha=.7,
                c='steelblue', s=60, edgecolor='white',
                label=f'GraphSAGE (R²={gs_metrics_reg["Test"]["R2"]:.3f})')
axes[0].scatter(y_act, pred_mlp_reg[test_mask].detach().numpy(), alpha=.7,
                c='darkorange', s=60, marker='s', edgecolor='white',
                label=f'MLP (R²={mlp_metrics_reg["Test"]["R2"]:.3f})')
lims = [y_act.min() - .5, y_act.max() + .5]
axes[0].plot(lims, lims, 'k--', alpha=.5)
axes[0].set(xlabel='Actual log(adoption)', ylabel='Predicted',
            title='Test Set: GraphSAGE vs MLP (Regression)')
axes[0].legend()

met_names = ['Reg R²', 'Cls AUC', 'Cls F1']
gs_vals  = [gs_metrics_reg['Test']['R2'],
            gs_metrics_cls['Test']['AUC'],
            gs_metrics_cls['Test']['F1']]
mlp_vals = [mlp_metrics_reg['Test']['R2'],
            mlp_metrics_cls['Test']['AUC'],
            mlp_metrics_cls['Test']['F1']]
xp = np.arange(len(met_names))
w  = 0.35
axes[1].bar(xp - w/2, gs_vals,  w, label='GraphSAGE', color='steelblue')
axes[1].bar(xp + w/2, mlp_vals, w, label='MLP',       color='darkorange')
axes[1].set_xticks(xp)
axes[1].set_xticklabels(met_names)
axes[1].set_ylabel('Score')
axes[1].set_title('Test-Set Metrics Comparison')
axes[1].legend()
axes[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

## 12.9 – City Embeddings & Adoption Predictions

In [ ]:
model_reg.eval()
with torch.no_grad():
    embeddings = model_reg.get_embeddings(x, edge_index).numpy()
embeddings = np.nan_to_num(embeddings, nan=0.0, posinf=0.0, neginf=0.0)

pca = PCA(n_components=2, random_state=42)
emb_2d = pca.fit_transform(embeddings)

city_regions = city_attr_dedup.set_index('city')['region'].to_dict()
region_colors = {
    'North America': '#e41a1c', 'Europe': '#377eb8', 'East Asia': '#4daf4a',
    'South Asia': '#984ea3', 'Southeast Asia': '#ff7f00', 'Middle East': '#a65628',
    'Oceania': '#f781bf', 'Africa': '#999999', 'Latin America': '#66c2a5',
}

pred_vals = pred_reg.detach().numpy()
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

sc = axes[0].scatter(emb_2d[:, 0], emb_2d[:, 1], c=pred_vals,
                      cmap='YlOrRd', s=60, edgecolor='white', linewidth=.5)
plt.colorbar(sc, ax=axes[0], label='Predicted log(adoption)')
for i, city in enumerate(all_cities):
    if (pred_vals[i] > np.percentile(pred_vals, 85) or
            pred_vals[i] < np.percentile(pred_vals, 15)):
        axes[0].annotate(city, (emb_2d[i, 0], emb_2d[i, 1]), fontsize=6, alpha=.8)
axes[0].set_title('GraphSAGE Embeddings – Predicted Adoption Intensity')
axes[0].set(xlabel='PC 1', ylabel='PC 2')

for i, city in enumerate(all_cities):
    r = city_regions.get(city, 'Other')
    axes[1].scatter(emb_2d[i, 0], emb_2d[i, 1],
                     c=region_colors.get(r, 'grey'),
                     s=60, edgecolor='white', linewidth=.5)
    if pred_vals[i] > np.percentile(pred_vals, 90):
        axes[1].annotate(city, (emb_2d[i, 0], emb_2d[i, 1]), fontsize=6, alpha=.8)
for r, c in region_colors.items():
    axes[1].scatter([], [], c=c, label=r, s=40)
axes[1].legend(fontsize=8, loc='best')
axes[1].set_title('GraphSAGE Embeddings – Colored by Region')
axes[1].set(xlabel='PC 1', ylabel='PC 2')

plt.tight_layout()
plt.show()
print(f'PCA explained variance: {pca.explained_variance_ratio_.sum():.1%}')

In [ ]:
result = pd.DataFrame({
    'city': all_cities,
    'predicted_log_adopt': pred_vals,
    'actual_log_adopt': y_reg.numpy(),
    'actual_test_count': [test_counts.get(c, 0) for c in all_cities],
    'is_test': test_mask.numpy(),
}).merge(city_attr_dedup[['city', 'region']], on='city', how='left').fillna(0)

result['pred_rank']   = result['predicted_log_adopt'].rank(ascending=False).astype(int)
result['actual_rank'] = result['actual_log_adopt'].rank(ascending=False).astype(int)

print('Top 20 cities by PREDICTED adoption intensity (test period):')
display(result.nlargest(20, 'predicted_log_adopt')[
    ['city', 'region', 'pred_rank', 'actual_rank',
     'predicted_log_adopt', 'actual_test_count']
].round(3))

rho, p_val = spearmanr(result['pred_rank'], result['actual_rank'])
print(f'\nRank correlation (all cities): Spearman ρ = {rho:.3f}, p = {p_val:.6f}')

test_result = result[result['is_test']]
rho_t, p_t = spearmanr(test_result['pred_rank'], test_result['actual_rank'])
print(f'Rank correlation (test only):  Spearman ρ = {rho_t:.3f}, p = {p_t:.6f}')

## 12.10 – Summary

In [ ]:
print('=' * 70)
print('STEP 12 SUMMARY: GraphSAGE – Predicting Adoption Intensity (Revised)')
print('=' * 70)
print(f'\n  Graph: {n_cities} nodes, {edge_index.shape[1]//2} undirected edges')
print(f'  Node features: {x.shape[1]} dims  '
      f'({len(exo_cols)} exogenous + {len(activity_names)} train-period activity)')
print(f'  Temporal split: train features ≤ {CUTOFF}, test target > {CUTOFF}')
print(f'  Node split: Train {train_mask.sum()}, Val {val_mask.sum()}, Test {test_mask.sum()}')
print(f'  Regularisation: dropout={DROPOUT}, weight_decay={WD}, '
      f'early_stop patience={PATIENCE}')

print(f'\n  {"─"*60}')
print(f'  Model G1 (Regression: log adoption count)')
print(f'    GraphSAGE   — Train R²={gs_metrics_reg["Train"]["R2"]:.4f}  '
      f'Val R²={gs_metrics_reg["Val"]["R2"]:.4f}  '
      f'Test R²={gs_metrics_reg["Test"]["R2"]:.4f}  '
      f'Test RMSE={gs_metrics_reg["Test"]["RMSE"]:.4f}')
print(f'    MLP baseline — Train R²={mlp_metrics_reg["Train"]["R2"]:.4f}  '
      f'Val R²={mlp_metrics_reg["Val"]["R2"]:.4f}  '
      f'Test R²={mlp_metrics_reg["Test"]["R2"]:.4f}  '
      f'Test RMSE={mlp_metrics_reg["Test"]["RMSE"]:.4f}')
print(f'    → ΔR² (GraphSAGE − MLP) = '
      f'{gs_metrics_reg["Test"]["R2"] - mlp_metrics_reg["Test"]["R2"]:+.4f}')

print(f'\n  {"─"*60}')
print(f'  Model G2 (Classification: high vs low adopter)')
print(f'    GraphSAGE   — Test AUC={gs_metrics_cls["Test"]["AUC"]:.4f}  '
      f'F1={gs_metrics_cls["Test"]["F1"]:.4f}')
print(f'    MLP baseline — Test AUC={mlp_metrics_cls["Test"]["AUC"]:.4f}  '
      f'F1={mlp_metrics_cls["Test"]["F1"]:.4f}')
print(f'    → ΔAUC = {gs_metrics_cls["Test"]["AUC"] - mlp_metrics_cls["Test"]["AUC"]:+.4f}')

print(f'\n  {"─"*60}')
print(f'  Rank correlation: ρ = {rho:.3f} (all), ρ = {rho_t:.3f} (test only)')

print(f'\n  {"─"*60}')
print('  Key takeaways:')
print('  1. Feature leakage fixed: all activity features computed from train period only.')
gs_r2 = gs_metrics_reg['Test']['R2']
mlp_r2 = mlp_metrics_reg['Test']['R2']
diff = gs_r2 - mlp_r2
if diff > 0.05:
    print(f'  2. GraphSAGE outperforms MLP (ΔR²={diff:+.4f}): graph structure adds value.')
elif diff < -0.05:
    print(f'  2. MLP outperforms GraphSAGE (ΔR²={diff:+.4f}): graph propagation may')
    print(f'     hurt with only {n_cities} nodes — the network is too small for GNN gains.')
else:
    print(f'  2. GraphSAGE ≈ MLP (ΔR²={diff:+.4f}): graph structure provides limited')
    print(f'     additional value beyond node features for {n_cities}-node network.')
print('  3. Early stopping & increased regularisation reduce overfitting risk.')
print('  4. City embeddings reveal latent groupings aligned with regional')
print('     geography and collaborative roles.')